In [1]:
import utils
import networkx as nx
import pandas as pd
import os
import requests
from pathlib import Path

In [2]:
# compute the common graph and the differences for each file

NETWORK_FOLDER = "../data/NetworksFromUp-InBetween1/"

def common_graph_and_extra(list_of_graphs):
    # make the networks
    list_files = {f"{pid}.txt" for pid in list_of_graphs}

    dict_of_graphs = {
        file: nx.read_edgelist(
            os.path.join(NETWORK_FOLDER, file),
            delimiter=";",
            nodetype=str,
            create_using=nx.DiGraph()
            )
        for file in os.listdir(NETWORK_FOLDER)
        if file in list_files
    }
    
    #common nodes and edges
    common_nodes = set.intersection(
        *(set(G.nodes()) for G in dict_of_graphs.values())
        )

    common_edges = set.intersection(
        *(set(G.edges()) for G in dict_of_graphs.values())
        )

    G_common = nx.DiGraph()
    G_common.add_nodes_from(common_nodes)
    G_common.add_edges_from(common_edges)

    #find differences
    graph_differences = {}

    for key, G in dict_of_graphs.items():
        graph_differences[key] = {
            "extra_nodes": set(G.nodes()) - common_nodes,
            "extra_edges": set(G.edges()) - common_edges
        }

    return graph_differences,common_nodes
    
graph_differences_primary,common_nodes_primary=common_graph_and_extra(utils.list_primary)    
graph_differences_recurrent,common_nodes_recurrent=common_graph_and_extra(utils.list_recurrent) 

In [3]:
path= "../data/extra_nodes"
path_set = Path(path)
path_set.mkdir(exist_ok=True)
os.chdir(path_set)

for keys, sets in graph_differences_primary.items():
    if sets:
        with open(f"{keys}", "w") as f:
            f.write("\n".join(sets["extra_nodes"]))
    

In [4]:
# find the pathways for the common graphs
path = ".."
path_set = Path(path)
path_set.mkdir(exist_ok=True)
os.chdir(path_set)

utils.enrichent_results(common_nodes_primary, "graph_common_pathways_primary", 'Reactome_Pathways_2024')
#utils.enrichent_results(common_nodes_recurrent, "graph_common_pathways_recurrent", 'Reactome_Pathways_2024')